# Dengesiz Veri Kümeleri ile Çalışma: SMOTE ve Sınıf Ağırlıkları (Class Weights)

Bu modül; gerçek hayat veri bilimi problemlerinde (sahtekarlık tespiti - Fraud Detection, nadir hastalık teşhisi, müşteri kaybı - Churn) en sık karşılaşılan **Sınıf Dengesizliği (Class Imbalance)** problemini, Doğruluk Paradoksunu (Accuracy Paradox) ve çözüm stratejilerini (**SMOTE ve Algoritmik Sınıf Ağırlıkları**) inceler.

---

## 1. Doğruluk Paradoksu (Accuracy Paradox) Nedir?

Bir veri kümesinde %99 negatif ($y=0$) ve %1 pozitif ($y=1$) sınıf varsa:
Tüm örneklere hiç öğrenmeden '0' diyen kör bir model **%99 Doğruluk (Accuracy)** elde eder!
Ancak pozitif sınıf için **Duyarlılık (Recall) = %0**'dır. Yani tüm dolandırıcılık veya hastalık vakaları kaçırılmıştır.

Bu nedenle dengesiz veri kümelerinde **Doğruluk (Accuracy) metrik olarak kullanılamaz**; yerine:
- **Hassasiyet (Precision):** $\frac{TP}{TP + FP}$
- **Duyarlılık (Recall):** $\frac{TP}{TP + FN}$
- **F1-Skoru:** $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$
- **PR-AUC (Precision-Recall Eğrisi Altındaki Alan)** kullanılmalıdır.

---

## 2. SMOTE (Synthetic Minority Over-sampling Technique) Matematiği

Chawla ve arkadaşları (2002) tarafından önerilen SMOTE; azınlık sınıfını rastgele kopyalamak (overfitting riski) yerine, uzayda sentetik yeni örnekler türetir:
1. Azınlık sınıfından bir $x_i$ örneği seçilir.
2. $x_i$'nin en yakın $k$-komşusu bulunur ($k=5$).
3. Komşulardan rastgele biri ($x_{zi}$) seçilir.
4. İki nokta arasına doğrusal enterpolasyonla yeni sentetik nokta yerleştirilir:
   $$x_{\text{new}} = x_i + \lambda \times (x_{zi} - x_i), \quad \lambda \sim U(0, 1)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

# 1. Aşırı Dengesiz Sentetik Veri Kümesi Üretimi (%95 Negatif, %5 Pozitif)
X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=6,
    weights=[0.95, 0.05],
    random_state=42
)

unique, counts = np.unique(y, return_counts=True)
print("Sınıf Dağılımı:")
for u, c in zip(unique, counts):
    print(f"  Sınıf {u}: {c} örnek (%{c/len(y)*100:.1f})")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


## 2. Deney 1: Dengesiz Veride Ham Model Eğitimi (Baseline)

In [ ]:
model_baseline = RandomForestClassifier(random_state=42)
model_baseline.fit(X_train, y_train)
y_pred_baseline = model_baseline.predict(X_test)

print("=== BASELINE MODEL RAPORU ===")
print(classification_report(y_test, y_pred_baseline, target_names=['Normal (0)', 'Nadir Olay (1)']))


## 3. Deney 2: SMOTE ile Sentetik Veri Artırma

In [ ]:
# DİKKAT: SMOTE SADECE eğitim kümesine uygulanmalıdır! Test kümesine ASLA uygulanmaz!
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

unique_smote, counts_smote = np.unique(y_train_smote, return_counts=True)
print(f"SMOTE Sonrası Eğitim Kümesi Sınıf Dağılımı: {dict(zip(unique_smote, counts_smote))}")

model_smote = RandomForestClassifier(random_state=42)
model_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = model_smote.predict(X_test)

print("=== SMOTE MODEL RAPORU ===")
print(classification_report(y_test, y_pred_smote, target_names=['Normal (0)', 'Nadir Olay (1)']))


## 4. Deney 3: Algoritmik Sınıf Ağırlıkları (Class Weights)

In [ ]:
# Veri türetmeden ceza matrisini azınlık sınıfı lehine ağırlıklandırma
model_weighted = RandomForestClassifier(class_weight='balanced', random_state=42)
model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)

print("=== BALANCED CLASS WEIGHTS RAPORU ===")
print(classification_report(y_test, y_pred_weighted, target_names=['Normal (0)', 'Nadir Olay (1)']))


## 5. Karışıklık Matrislerinin (Confusion Matrix) Karşılaştırmalı Analizi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm_base = confusion_matrix(y_test, y_pred_baseline)
ConfusionMatrixDisplay(cm_base, display_labels=['0', '1']).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title("1. Baseline (Dengesiz)")

cm_smote = confusion_matrix(y_test, y_pred_smote)
ConfusionMatrixDisplay(cm_smote, display_labels=['0', '1']).plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title("2. SMOTE Uygulanmış")

cm_weight = confusion_matrix(y_test, y_pred_weighted)
ConfusionMatrixDisplay(cm_weight, display_labels=['0', '1']).plot(ax=axes[2], cmap='Oranges', colorbar=False)
axes[2].set_title("3. Class Weight='balanced'")

plt.tight_layout()
plt.show()


## 6. Mühendislik Çıkarımları

1. **Veri Sızıntısı (Data Leakage) Kuralı:** SMOTE asla `train_test_split` öncesinde uygulanmaz. Test kümesine SMOTE uygulanırsa sentetik noktalar test verisine sızar ve model yapay yüksek başarı gösterir.
2. **SMOTE vs Class Weights:** Yüksek boyutlu seyrek verilerde (örn. metin) SMOTE gürültüyü artırabilir; bu durumlarda `class_weight='balanced'` daha kararlıdır.
3. **Nihai Başarı Metriği:** Nadir olaylarda (dolandırıcılık, arıza tespiti) False Negative maliyeti çok yüksek olduğu için **Recall** ve **PR-AUC** temel karar metriğidir.
